In [ ]:
import torch
from torch import nn

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()

embedding = nn.Embedding(200021, 128, device=DEVICE)

def alloc_fn(size: int, align: int, _):
  return torch.empty(size, dtype=torch.int8aaq
                     , device=DEVICE)

triton.set_allocator(alloc_fn)

@triton.jit
def _attention_bwd_pre_process(o_ptr, do_ptr, delta_ptr,
                               n_ctx: tl.constexpr,
                               pre_block: tl.constexpr,
                               heads: tl.constexpr,
                               hidden: tl.constexpr):
  pre_block_per_nctx = tl.program_id(0)
  off_h = tl.program_id(1)
  offs_pre_block = pre_block_per_nctx*pre_block + tl.arange(0, pre_block)
  offs_hid = tl.arange(0, hidden)
  offset = off_h*n_ctx*hidden + offs_pre_block[:,None]*hidden + offs_hid[None,:]
  o = tl.load(o_ptr + offset)
  do = tl.load(do_ptr + offset)
  o_do = tl.sum(o*do, axis=1)
  delta = delta_ptr + off_h*n_ctx + offs_pre_block
  tl.store(delta, o_do)


@triton.jit
def _attention_bwd(q, k, v, do, dq, dk, dv, m, d,
                   sm_scale: tl.constexpr, num_heads: tl.constexpr,
                   n_ctx: tl.constexpr, hidden_dim: tl.constexpr, block_m: tl.constexpr,
                   block_n: tl.constexpr, bulk_slice_factor: tl.constexpr):
  LN2 = 0.6931471824645996  # = ln(2)
  # current context block
  ctxid = tl.program_id(0)
  # current head
  hid = tl.program_id(1)
  # init offset can be used for both dkdv & dq
  init_offset = ctxid*block_m + hid*n_ctx
  y_dim = num_heads * n_ctx

  desc_v = tl.make_tensor_descriptor(v, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                        block_shape=[block_m, hidden_dim])
  desc_k = tl.make_tensor_descriptor(k, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                    block_shape=[block_m, hidden_dim])
  

  desc_dv = tl.make_tensor_descriptor(dv, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                        block_shape=[block_m, hidden_dim])
  desc_dk = tl.make_tensor_descriptor(dk, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                    block_shape=[block_m, hidden_dim])
  desc_dq = tl.make_tensor_descriptor(dq, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                        block_shape=[block_m, hidden_dim])
  
  dvalue = tl.zeros([block_m, hidden_dim], dtype=tl.float32)
  dkey = tl.zeros([block_m, hidden_dim], dtype=tl.float32)
  dquery = tl.zeros([block_m, hidden_dim], dtype=tl.float32)

  mask_block_n:tl.constexpr = block_n // bulk_slice_factor
  key = desc_k.load([init_offset,0])
  value = desc_v.load([init_offset,0])

  desc_query_mask_block_n = tl.make_tensor_descriptor(q, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                          block_shape=[mask_block_n, hidden_dim])
  desc_do_mask_block_n = tl.make_tensor_descriptor(do, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                    block_shape=[mask_block_n, hidden_dim])
  desc_key_mask_block_n = tl.make_tensor_descriptor(k, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                    block_shape=[mask_block_n, hidden_dim])
  offset_m = init_offset + tl.arange(0, block_m)
  offset = init_offset
  # Mask
  for sub_block_n in tl.range(0, block_m, mask_block_n):
    offset += sub_block_n
    offset_n = offset + tl.arange(0, mask_block_n)
    dquery_temp, dkey_temp, dvalue_temp = _attention_bwd_dqdkdv_per_sublock_n(dquery, dkey, dvalue, m, d,
                                      key, value, desc_query_mask_block_n,
                                      desc_do_mask_block_n, desc_key_mask_block_n, True, offset_m, offset_n, offset)
    dkey += dkey_temp
    dvalue += dvalue_temp
    dquery += dquery_temp

  # right of mask for non mask regions
  desc_query_block_n = tl.make_tensor_descriptor(q, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                          block_shape=[block_n, hidden_dim])
  desc_do_block_n = tl.make_tensor_descriptor(do, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                    block_shape=[block_n, hidden_dim])
  desc_key_block_n = tl.make_tensor_descriptor(k, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                    block_shape=[block_n, hidden_dim])

  diff: tl.constexpr = n_ctx-(ctxid*block_m + block_m)
  offset =  init_offset + block_m
  for sub_block_n in tl.range(0, diff, block_n):
    offset += sub_block_n
    offset_n = offset + tl.arange(0, block_n)
    dquery_temp, dkey_temp, dvalue_temp = _attention_bwd_dqdkdv_per_sublock_n(dquery, dkey, dvalue, m, d,
                                      key, value, desc_query_block_n,
                                      desc_do_block_n, desc_key_block_n, False, offset_m, offset_n, offset)
    dkey += dkey_temp
    dvalue += dvalue_temp
    dquery += dquery_temp

  desc_dv.store([init_offset, 0], dvalue)
  desc_dk.store([init_offset, 0], dkey*sm_scale)
  desc_dq.store([init_offset, 0], dquery*LN2)


@triton.jit
def _attention_bwd_dqdkdv_per_sublock_n(dquery, dkey, dvalue, m, d,
                                      key, value, desc_query_mask_block_n, desc_do_mask_block_n,
                                      desc_key_mask_block_n, MASK, offset_m, offset_n, offset):
  query = desc_query_mask_block_n.load([offset,0])
  key_n = desc_key_mask_block_n.load([offset,0])
  max_tensor = tl.load(m+offset_n)
  qkT = tl.dot(key, tl.trans(query))
  pT = tl.math.exp2(qkT - max_tensor[None, :])
  if MASK:
    mask = (offset_n[None, :] >= offset_m[:, None])
    pT = tl.where(mask, pT, 0.0)
  do = desc_do_mask_block_n.load([offset, 0]).to(tl.bfloat16)
  dvalue += tl.dot(pT.to(tl.bfloat16), do)
  delta = tl.load(d+offset_n)
  doT = tl.trans(do)
  dpT = tl.dot(value.to(tl.bfloat16), doT).to(tl.float32)
  dsT = pT * (dpT - delta[None, :])
  dsT = dsT.to(tl.bfloat16)
  dquery += tl.dot(dsT, key_n.to(tl.bfloat16))
  dkey += tl.dot(dsT, query.to(tl.bfloat16))
  return dquery, dkey, dvalue

num_heads = 8
embedded_tensor = embedding(torch.randint(low=0, high=200021, size=(2112,), device=DEVICE))
q = embedded_tensor.unsqueeze(0).expand(num_heads, -1, -1)
k = embedded_tensor.unsqueeze(0).expand(num_heads, -1, -1)
v = embedded_tensor.unsqueeze(0).expand(num_heads, -1, -1)
o = torch.rand_like(q)
do = torch.rand_like(q)
sm_scale = 1.3
block_m = 32
block_n = 16

pre_block = 128
num_hiddens = q.shape[-1]
n_ctx = q.shape[1]
M = torch.randn((q.shape[0], q.shape[1]), device=q.device, dtype=torch.float32)
grid = (q.shape[1]//pre_block, num_heads, 1)
print(f"Grid : {grid}, q: {q.shape}, k: {k.shape}, v: {v.shape} ")
delta = torch.randn((q.shape[0], q.shape[1]), device=q.device, dtype=torch.float32)
# Preprocess
_attention_bwd_pre_process[grid](o, do, delta, n_ctx, pre_block, num_heads, num_hiddens)
# print(f"Delta : {delta}")
dq = torch.empty_like(q)
dk = torch.empty_like(k)
dv = torch.empty_like(v)
bulk_slice_factor = 1
grid_bwd = (n_ctx//block_m, num_heads, 1)
print(f"Grid (bwd) : {grid_bwd}")
_attention_bwd[grid_bwd](q, k, v, do, dq, dk, dv, M, delta, sm_scale, num_heads, n_ctx,
                          num_hiddens, block_m, block_n, bulk_slice_factor)
print(f"dv : {dv}")
print(f"dk : {dk}")
print(f"dq : {dq}")